# Assignment Beta - Question 2: LoRA vs QLoRA vs DoRA vs AdaLoRA for function calling
Student 23K-0003. **Run this notebook on a GPU runtime (Colab T4 16 GB is enough; ~1-2 h total).**
Nothing in this notebook has been executed yet - all result cells are produced when YOU run it, and the report template (Assignment_Beta_Question_no_2.docx) must be filled with those real outputs.

Controlled comparison: same base model, same 2,000-example slice (1,800 train / 200 held-out test), same seed, epochs, LR schedule and target modules. Only the adaptation method changes.


## Section 1 - Theory calculations (pure Python)

In [ ]:

d_in = d_out = 4096; r = 16
lora = r*(d_in + d_out)           # A: r x d_in, B: d_out x r
full = d_in*d_out
print("(a) trainable params = r*(d_in+d_out) = 16*(4096+4096) =", lora)
print("(b) fraction of layer =", lora, "/", full, "=", f"{100*lora/full:.5f}%")


(a) trainable params = r*(d_in+d_out) = 16*(4096+4096) = 131072
(b) fraction of layer = 131072 / 16777216 = 0.78125%


### (c) Why B = 0 and A random?
The adapted layer is W = W0 + (alpha/r) B A. With **B = 0** the product BA is exactly 0, so at step 0 the adapted model is *identical* to the pretrained model: training starts from the known-good function, no noise is injected, and the loss at step 0 equals the base model's loss.
**A must be random (non-zero)** because the gradient to B is dL/dB = (dL/dW) A^T; if A were also zero this gradient would be zero and B could never move (and the gradient to A, B^T (dL/dW), would also be zero because B = 0) - a dead start / saddle point. Random A makes dL/dB non-zero, B becomes non-zero after the first update, and then A starts receiving gradient too.
**If both were random**, BA would be a random matrix of non-trivial norm: on the very first forward pass W0 x would be corrupted by (alpha/r)BAx, the model's outputs would be perturbed (loss spikes above the base loss, gradients are large and noisy), part of the pretrained knowledge is damaged before learning begins, and training becomes unstable / needs a smaller learning rate.

### (d) Why each variant exists
* **QLoRA** - plain LoRA still keeps the frozen base weights in 16-bit (1.5B params = ~3 GB; 7B = 14 GB). QLoRA stores the frozen weights in **4-bit NormalFloat (NF4)** - a data type whose 16 levels are the quantiles of a normal distribution, which is information-theoretically suited to normally distributed weights - with double quantization of the scale constants, and dequantizes blocks on the fly to bf16 for compute while gradients flow into 16-bit LoRA adapters. **Paged optimizers** (unified-memory paged AdamW) move optimizer state to CPU RAM when GPU memory spikes (e.g. long sequences) so training does not crash with OOM. Problem solved: *memory*, so a big model can be fine-tuned on one small GPU, at some cost in speed and a small accuracy gap.
* **DoRA** - decomposes each weight into **magnitude** (a per-column norm vector m) and **direction** (V / ||V||), W = m * (W0 + BA)/||W0 + BA||_c. LoRA is applied only to the direction while m is trained directly. Full fine-tuning tends to change magnitude and direction in a decoupled way whereas LoRA couples them; DoRA recovers that learning pattern and usually closes part of the accuracy gap to full fine-tuning at the same (or slightly higher) rank, and is more robust to the choice of rank - at the cost of extra compute/memory for the norm and unmerged inference overhead.
* **AdaLoRA** - plain LoRA gives every matrix the same rank, although some layers/matrices matter more. AdaLoRA parametrizes the update in SVD form P Lambda Q (with orthogonality regularization) and **scores the importance of each singular value** (sensitivity-based importance of the triplets), then **prunes low-importance singular values** during training, so a global rank budget (start r=12 -> target r=8 average) is allocated where it is actually needed. Problem solved: *wasted rank / budget allocation*.


## Section 2 - Setup

In [ ]:

!pip -q install -U "transformers>=4.46" "peft>=0.13" "bitsandbytes>=0.43" accelerate datasets matplotlib


In [ ]:

import os, re, json, time, random, gc, math, copy
import torch, numpy as np
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments, DataCollatorForSeq2Seq)
from peft import LoraConfig, AdaLoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

SEED = 42
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
BASE = "Qwen/Qwen2.5-1.5B-Instruct"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16   # T4 has no bf16 -> fp16
EPOCHS, LR, BS, GA, MAXLEN = 2, 2e-4, 4, 4, 768
TARGETS = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
print("device:", torch.cuda.get_device_name(0), "| dtype:", DTYPE)
tok = AutoTokenizer.from_pretrained(BASE); tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token


device: Tesla T4 | dtype: torch.bfloat16


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

### Data: glaive-function-calling-v2 - 2,000 examples, 200 held out (never used in training or model selection except final evaluation)

In [ ]:

ds = load_dataset("glaiveai/glaive-function-calling-v2", split="train")
CALL_RE = re.compile(r"<functioncall>\s*(\{.*?\})\s*(?:<\|endoftext\|>|$)", re.S)
USER_RE = re.compile(r"USER:\s*(.*?)\s*(?:\n\s*\n\s*A:|ASSISTANT:|$)", re.S)

def parse_call(txt):
    """glaive stores {"name": "...", "arguments": '{"a": 1}'} (arguments as a single-quoted JSON string)."""
    m = re.search(r'"name":\s*"([^"]+)"\s*,\s*"arguments":\s*(.*)\}\s*$', txt.strip(), re.S)
    if not m: return None
    name, args = m.group(1), m.group(2).strip()
    if args[:1] == "'" and args[-1:] == "'": args = args[1:-1]
    try: return {"name": name, "arguments": json.loads(args)}
    except Exception: return None

examples = []
for row in ds:
    chat, system = row["chat"], row["system"]
    if "<functioncall>" not in chat: continue
    um = USER_RE.search(chat)
    cm = CALL_RE.search(chat)
    if not um or not cm: continue
    call = parse_call(cm.group(1))
    if call is None: continue
    examples.append({"system": system.replace("SYSTEM:", "").strip(), "user": um.group(1).strip(), "target": call})
    if len(examples) >= 2000: break
print(len(examples), "usable examples"); print(examples[0])
rng = random.Random(SEED); rng.shuffle(examples)
test, train = examples[:200], examples[200:]          # 200 held-out test examples, 1,800 train
json.dump(test, open("test_set.json","w"))
INSTR = "You are a tool-calling assistant. Reply with ONLY a JSON object {\"name\": ..., \"arguments\": {...}} choosing one of the functions below. No other text.\n\n"
def prompt_of(ex):
    msgs = [{"role":"system","content": INSTR + ex["system"]}, {"role":"user","content": ex["user"]}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
def encode(ex):
    p = tok(prompt_of(ex), add_special_tokens=False)["input_ids"]
    t = tok(json.dumps(ex["target"]) + tok.eos_token, add_special_tokens=False)["input_ids"]
    ids = (p + t)[:MAXLEN]; labels = ([-100]*len(p) + t)[:MAXLEN]       # loss only on the answer
    return {"input_ids": ids, "attention_mask":[1]*len(ids), "labels": labels}
from datasets import Dataset
train_ds = Dataset.from_list([encode(ex) for ex in train])
print(train_ds)


2000 usable examples
{'system': 'You are a helpful assistant with access to the following functions. Use them if required -\n{\n    "name": "get_news_headlines",\n    "description": "Get the latest news headlines",\n    "parameters": {\n        "type": "object",\n        "properties": {\n            "country": {\n                "type": "string",\n                "description": "The country for which to fetch news"\n            }\n        },\n        "required": [\n            "country"\n        ]\n    }\n}', 'user': 'Can you tell me the latest news headlines for the United States?', 'target': {'name': 'get_news_headlines', 'arguments': {'country': 'United States'}}}
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1800
})


### Evaluation: JSON validity, tool accuracy, full match (200 test examples, greedy decoding)

In [ ]:

@torch.no_grad()
def evaluate(model, bs=8, max_new=160):
    model.eval(); outs = []
    for i in range(0, len(test), bs):
        batch = test[i:i+bs]
        enc = tok([prompt_of(e) for e in batch], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    valid = name_ok = full = 0; records = []
    for e, o in zip(test, outs):
        try: j = json.loads(o.strip()); ok = isinstance(j, dict)
        except Exception: j, ok = None, False
        v = ok; valid += v
        n = v and j.get("name") == e["target"]["name"]; name_ok += n
        f = n and j.get("arguments") == e["target"]["arguments"]; full += f
        records.append({"user": e["user"], "gold": e["target"], "pred": o, "json_valid": bool(v), "tool_ok": bool(n), "full": bool(f)})
    return {"json_valid": valid, "tool_acc": name_ok, "full_match": full, "n": len(test)}, records


### Training helper - identical for every run except the adapter config

In [ ]:

def load_base(quant=False):
    kw = dict(torch_dtype=DTYPE, device_map={"":0})
    if quant:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    return AutoModelForCausalLM.from_pretrained(BASE, **kw)

def make_peft(method, r):
    quant = method == "qlora"
    model = load_base(quant)
    if quant: model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    else:
        model.gradient_checkpointing_enable(); model.enable_input_require_grads()
    steps_total = math.ceil(len(train_ds) / (BS*GA)) * EPOCHS
    if method in ("lora", "qlora"):
        cfg = LoraConfig(r=r, lora_alpha=2*r, lora_dropout=0.05, target_modules=TARGETS, task_type=TaskType.CAUSAL_LM)
    elif method == "dora":
        cfg = LoraConfig(r=r, lora_alpha=2*r, lora_dropout=0.05, target_modules=TARGETS, task_type=TaskType.CAUSAL_LM, use_dora=True)
    elif method == "adalora":
        cfg = AdaLoraConfig(init_r=12, target_r=8, lora_alpha=16, lora_dropout=0.05, target_modules=TARGETS, task_type=TaskType.CAUSAL_LM,
                            tinit=int(0.1*steps_total), tfinal=int(0.2*steps_total), deltaT=10, total_step=steps_total)
    model = get_peft_model(model, cfg)
    for n_, p_ in model.named_parameters():                     # adapters in fp32 for stable fp16/bf16 training
        if p_.requires_grad: p_.data = p_.data.float()
    return model, steps_total

def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    return tr, tot

def run(method, r, tag=None):
    seed_all(); gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    model, steps_total = make_peft(method, r)
    tr, tot = count_params(model)
    args = TrainingArguments(output_dir=f"out_{tag or method}", per_device_train_batch_size=BS, gradient_accumulation_steps=GA,
        num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type="cosine", warmup_steps=max(1, int(0.03*steps_total))
, logging_steps=20, save_strategy="no",
        bf16=(DTYPE==torch.bfloat16), fp16=(DTYPE==torch.float16), seed=SEED, report_to="none",
        optim="paged_adamw_8bit" if method=="qlora" else "adamw_torch", gradient_checkpointing=True, remove_unused_columns=False)
    class AdaTrainer(Trainer):                                   # AdaLoRA needs update_and_allocate after each optimizer step
        def training_step(self, *a, **k):
            loss = super().training_step(*a, **k)
            if method == "adalora": self.model.base_model.update_and_allocate(self.state.global_step)
            return loss
    trainer = AdaTrainer(model=model, args=args, train_dataset=train_ds,
                         data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
    t0 = time.time(); trainer.train(); minutes = (time.time()-t0)/60
    peak = torch.cuda.max_memory_allocated()/2**30
    metrics, records = evaluate(model)
    res = {"run": tag or method, "method": method, "rank": r, "trainable": tr, "total": tot, "trainable_pct": 100*tr/tot,
           "peak_gpu_GiB": round(peak,3), "train_minutes": round(minutes,2), **metrics}
    print(json.dumps(res, indent=1))
    json.dump(records, open(f"records_{tag or method}.json","w"))
    model.save_pretrained(f"adapter_{tag or method}")
    del model, trainer; gc.collect(); torch.cuda.empty_cache()
    return res
RESULTS = []


## Run 0 - base model, no fine-tuning (the 'before' picture)

In [ ]:

m0 = load_base(False); tr, tot = 0, sum(p.numel() for p in m0.parameters())
t0 = time.time(); met, rec = evaluate(m0);
res0 = {"run":"base","method":"none","rank":None,"trainable":0,"total":tot,"trainable_pct":0.0,"peak_gpu_GiB":None,"train_minutes":0, **met}
print(res0); json.dump(rec, open("records_base.json","w")); RESULTS.append(res0); del m0; gc.collect(); torch.cuda.empty_cache()


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'run': 'base', 'method': 'none', 'rank': None, 'trainable': 0, 'total': 1543714304, 'trainable_pct': 0.0, 'peak_gpu_GiB': None, 'train_minutes': 0, 'json_valid': 200, 'tool_acc': 197, 'full_match': 132, 'n': 200}


## Runs 1-4 (r=8; AdaLoRA start 12 -> target 8)

In [ ]:
!pip uninstall -y torchao


In [ ]:
RESULTS.append(run("lora", 8))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Step,Training Loss
20,0.123529
40,0.058960
60,0.052051
80,0.071049
100,0.041106
120,0.063301
140,0.035529
160,0.036018
180,0.038550
200,0.027912


{
 "run": "lora",
 "method": "lora",
 "rank": 8,
 "trainable": 9232384,
 "total": 1552946688,
 "trainable_pct": 0.5945074657965335,
 "peak_gpu_GiB": 11.33,
 "train_minutes": 97.4,
 "json_valid": 200,
 "tool_acc": 200,
 "full_match": 153,
 "n": 200
}


In [ ]:
RESULTS.append(run("qlora", 8))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3195: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Step,Training Loss
20,0.131074
40,0.063569
60,0.054077
80,0.075935
100,0.043031
120,0.066768
140,0.037776
160,0.038033
180,0.040676
200,0.029004


{
 "run": "qlora",
 "method": "qlora",
 "rank": 8,
 "trainable": 9232384,
 "total": 897848832,
 "trainable_pct": 1.028278221338712,
 "peak_gpu_GiB": 10.588,
 "train_minutes": 101.82,
 "json_valid": 200,
 "tool_acc": 200,
 "full_match": 151,
 "n": 200
}


In [ ]:
RESULTS.append(run("dora", 8))

In [ ]:
RESULTS.append(run("adalora", 8))

In [ ]:

import pandas as pd
df = pd.DataFrame(RESULTS); df["json_valid_%"] = 100*df.json_valid/df.n; df["tool_acc_%"] = 100*df.tool_acc/df.n; df["full_match_%"] = 100*df.full_match/df.n
df.to_csv("results_main.csv", index=False); df


## Section 3 - rank sweep for the best method (by tool accuracy; ties -> lower memory)

In [ ]:

best = df[df.method!="none"].sort_values(["tool_acc","peak_gpu_GiB"], ascending=[False, True]).iloc[0]
print("best method:", best.method)
SWEEP = []
for r in (4, 16, 32):
    if best.method == "adalora": print("AdaLoRA: target_r sweep not defined by the brief; using LoRA config with init_r/target_r = 1.5r/r would change two things - re-run as LoRA r if AdaLoRA wins"); break
    SWEEP.append(run(best.method, r, tag=f"{best.method}_r{r}"))
SWEEP.append({k: best[k] for k in ["run","method","rank","trainable","total","trainable_pct","peak_gpu_GiB","train_minutes","json_valid","tool_acc","full_match","n"]})
sw = pd.DataFrame(SWEEP).sort_values("rank"); sw["tool_acc_%"] = 100*sw.tool_acc/sw.n; sw.to_csv("results_rank_sweep.csv", index=False); sw


In [ ]:

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5.5,3.6)); ax.plot(sw["rank"], sw["tool_acc_%"], "o-"); ax.set_xscale("log", base=2)
ax.set_xticks(sw["rank"]); ax.set_xticklabels(sw["rank"]); ax.set_xlabel("LoRA rank r"); ax.set_ylabel("Tool accuracy (%) on 200 test examples")
ax.set_title(f"Tool accuracy vs rank ({best.method})"); ax.grid(alpha=.3); plt.tight_layout(); plt.savefig("rank_plot.png", dpi=200); plt.show()
print(sw[["rank","trainable","peak_gpu_GiB","train_minutes","tool_acc_%"]])


## Section 4 support - error analysis, merge behaviour, speed

In [ ]:

rec = json.load(open(f"records_{best.run}.json"))
bad = [r for r in rec if not r["full"]]
print(len(bad), "of 200 not fully correct. Five to discuss (pick diverse ones):\n")
for r in bad[:5]:
    print("USER :", r["user"][:300]); print("GOLD :", json.dumps(r["gold"])[:300]); print("PRED :", r["pred"][:300])
    print("label: json_valid=%s tool_ok=%s\n" % (r["json_valid"], r["tool_ok"]))


In [ ]:

# Merge test: LoRA/DoRA merge into bf16 weights; QLoRA adapters were trained against NF4-quantized weights, so merging into
# a dequantized/bf16 base changes the numerics (merge error). Measure it: accuracy of unmerged vs merged adapter.
from peft import PeftModel
def eval_merged(method_tag, quant):
    base = load_base(quant); m = PeftModel.from_pretrained(base, f"adapter_{method_tag}")
    unmerged, _ = evaluate(m)
    if quant: base = load_base(False); m = PeftModel.from_pretrained(base, f"adapter_{method_tag}")    # QLoRA: merge into 16-bit base
    t0=time.time(); merged_model = m.merge_and_unload(); merged, _ = evaluate(merged_model); dt=time.time()-t0
    print(method_tag, "unmerged:", unmerged, "| merged:", merged, f"| merged eval time {dt:.0f}s")
    del m, base, merged_model; gc.collect(); torch.cuda.empty_cache()
for tag, q in [("lora", False), ("qlora", True), ("dora", False)]: eval_merged(tag, q)
